# ARES learned-feature sparse reconstruction with HLoc

This notebook evaluates the existing 22 hostel-corridor photographs with HLoc, SuperPoint, LightGlue, exhaustive pairing, and HLoc's PyCOLMAP Structure-from-Motion integration. It does not run dense reconstruction or create a mesh.

**Important calibration choice:** the source images have no EXIF camera calibration. This run uses one shared `SIMPLE_RADIAL` camera initialized at `[1920, 800, 450, 0]` pixels/coefficients and disables bundle-adjustment refinement of focal length, principal point, and distortion. `1920 px` is COLMAP's 1.2 × 1600 default heuristic, not a measured focal length. The report marks calibration as fixed and unvalidated.


## 1. Check the Colab runtime and install HLoc

In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU), then run this cell. HLoc's official install instructions use a recursive clone and editable install; its requirements include PyTorch, PyCOLMAP, and LightGlue. Nothing is installed on the local laptop by this notebook.


In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("A Colab GPU runtime is required for this learned-feature experiment.")

HLOC_REPO = Path("/content/Hierarchical-Localization")
if not HLOC_REPO.exists():
    subprocess.run([
        "git", "clone", "--recursive",
        "https://github.com/cvg/Hierarchical-Localization.git", str(HLOC_REPO)
    ], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(HLOC_REPO)], check=True)
sys.path.insert(0, str(HLOC_REPO))

import hloc
import pycolmap
from hloc import extract_features, match_features, pairs_from_exhaustive, reconstruction

print("HLoc:", hloc.__version__)
print("PyCOLMAP:", pycolmap.__version__)


## 2. Local helpers for safe input, paths, and reporting

The ZIP must contain the same 22 unique JPG/JPEG/PNG/WEBP photographs. ZIP paths are checked before extraction, images are decoded with Pillow, and names are sorted deterministically. These helpers use only the standard library and Pillow.


In [ ]:
import hashlib
import json
import math
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath
from statistics import mean, median

from PIL import Image

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}


def safe_extract_image_zip(zip_path, destination, expected_count=22):
    destination = Path(destination)
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError(f"Extraction directory is not empty: {destination}")
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        candidates = []
        for info in archive.infolist():
            if info.is_dir():
                continue
            raw_name = info.filename.replace("\\", "/")
            member = PurePosixPath(raw_name)
            mode = info.external_attr >> 16
            if member.is_absolute() or ".." in member.parts or stat.S_ISLNK(mode):
                raise ValueError(f"Unsafe ZIP member: {info.filename}")
            if member.suffix.lower() in IMAGE_SUFFIXES and "__MACOSX" not in member.parts:
                candidates.append((info, member))
        if len(candidates) != expected_count:
            raise ValueError(f"Expected {expected_count} image files, found {len(candidates)}")
        hashes = set()
        names = set()
        for info, member in candidates:
            target = destination.joinpath(*member.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.exists():
                raise ValueError(f"Duplicate ZIP image path: {member.as_posix()}")
            target.write_bytes(archive.read(info))
            try:
                with Image.open(target) as image:
                    image.verify()
                with Image.open(target) as image:
                    if image.width < 32 or image.height < 32:
                        raise ValueError(f"Image is too small: {member.as_posix()}")
            except Exception as error:
                raise ValueError(f"Invalid image {member.as_posix()}: {error}") from error
            digest = hashlib.sha256(target.read_bytes()).hexdigest()
            if digest in hashes:
                raise ValueError(f"Duplicate image contents: {member.as_posix()}")
            hashes.add(digest)
            names.add(member.as_posix())
    ordered = sorted(names)
    if len(ordered) != expected_count:
        raise ValueError("Image names are not unique.")
    return ordered


def create_output_paths(root):
    root = Path(root)
    paths = {name: root / name for name in ("features", "matches", "reconstruction", "sparse")}
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    (paths["sparse"] / "0").mkdir(parents=True, exist_ok=True)
    return paths


def validate_feature_preprocessing(config):
    model = config.get("model", {})
    preprocessing = config.get("preprocessing", {})
    if model.get("name") != "superpoint":
        raise ValueError("Expected HLoc's SuperPoint feature configuration")
    if preprocessing.get("grayscale") is not True or preprocessing.get("resize_max") != 1600:
        raise ValueError("Expected grayscale SuperPoint preprocessing at a 1600 px maximum")
    if model.get("max_keypoints") != 4096:
        raise ValueError("Expected the documented 4096-keypoint SuperPoint limit")
    return {"grayscale": True, "resize_max": 1600, "max_keypoints": 4096}


def build_reconstruction_report(model, image_names, baseline, calibration_refined, initial_camera_params):
    image_names = sorted(image_names)
    registered = []
    cameras = []
    positions = []
    point_errors = []
    track_lengths = []
    point_count = 0
    if model is not None:
        registered_ids = sorted(model.reg_image_ids())
        registered = sorted(model.images[image_id].name for image_id in registered_ids)
        for camera in model.cameras.values():
            params = [float(value) for value in camera.params]
            model_id = camera.model
            model_name = getattr(camera, "model_name", None) or getattr(model_id, "name", str(model_id))
            cameras.append({
                "camera_id": int(camera.camera_id), "model": str(model_name),
                "width": int(camera.width), "height": int(camera.height),
                "parameters": params,
                "focal_length_px": params[0] if params else None,
                "principal_point_px": params[1:3] if len(params) >= 3 else None,
                "distortion_parameters": params[3:],
                "calibration_refined": bool(calibration_refined),
            })
        for image_id in registered_ids:
            image = model.images[image_id]
            center = image.projection_center()
            positions.append({"image": image.name, "x": float(center[0]), "y": float(center[1]), "z": float(center[2])})
        for point in model.points3D.values():
            point_count += 1
            error = float(point.error)
            if math.isfinite(error):
                point_errors.append(error)
            track_lengths.append(int(point.track.length()))
    return {
        "input_images": len(image_names), "registered_images": len(registered),
        "registered_image_names": registered,
        "unregistered_images": sorted(set(image_names) - set(registered)),
        "points_3d": point_count, "camera_count": len(cameras), "cameras": cameras,
        "mean_reprojection_error_px": mean(point_errors) if point_errors else None,
        "median_reprojection_error_px": median(point_errors) if point_errors else None,
        "mean_track_length": mean(track_lengths) if track_lengths else None,
        "camera_positions": positions,
        "calibration_policy": {
            "shared_camera": True, "model": "SIMPLE_RADIAL",
            "initial_parameters": list(initial_camera_params),
            "refined": bool(calibration_refined),
            "note": "Initial parameters are a heuristic, not measured calibration; interpret camera fit cautiously.",
        },
        "baseline": baseline,
        "assessment": "Compare registration, reprojection error, track length, calibration plausibility, and spatial coverage; point count alone is insufficient.",
    }


def save_reconstruction_report(report, report_path):
    report_path = Path(report_path)
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return report_path


## 3. Upload and validate the existing photo ZIP

Upload the ready-made `data/terrain_dataset/ares_hloc_photos.zip` from the project workspace below. It contains the existing 22 source photographs and is ignored by Git. If rebuilding it, ZIP only the contents of `data/terrain_dataset/raw/`; do not include the reconstruction database or generated files.


In [ ]:
from google.colab import files

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError(f"Upload exactly one ZIP file; received {zip_names}")

OUTPUT_ROOT = Path("/content/ares_hloc_outputs")
IMAGE_DIR = Path("/content/ares_hloc_input_images")
if IMAGE_DIR.exists():
    shutil.rmtree(IMAGE_DIR)
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)  # start a clean, isolated HLoc experiment output
image_names = safe_extract_image_zip(zip_names[0], IMAGE_DIR, expected_count=22)
paths = create_output_paths(OUTPUT_ROOT)
print(f"Validated {len(image_names)} unique images")
print("First/last filenames:", image_names[0], "…", image_names[-1])
print("Output paths:", {key: str(value) for key, value in paths.items()})


## 4. Extract SuperPoint and match all image pairs with LightGlue

Twenty-two images yield 231 exhaustive unordered pairs. HLoc's documented exhaustive-pair generator writes the pair list; its standard `superpoint_max` and `superpoint+lightglue` configurations perform learned extraction and matching. `reconstruction.main` performs geometric verification unless explicitly told to skip it; this notebook keeps verification enabled.


In [ ]:
sfm_pairs = paths["matches"] / "pairs-exhaustive.txt"
sfm_dir = paths["reconstruction"]

feature_conf = extract_features.confs["superpoint_max"]
matcher_conf = match_features.confs["superpoint+lightglue"]
preprocessing = validate_feature_preprocessing(feature_conf)
assert matcher_conf["model"]["name"] == "lightglue"

feature_path = extract_features.main(
    feature_conf, IMAGE_DIR, paths["features"], image_list=image_names, overwrite=True
)
pairs_from_exhaustive.main(sfm_pairs, image_list=image_names)
pair_count = len(sfm_pairs.read_text(encoding="utf-8").splitlines())
assert pair_count == len(image_names) * (len(image_names) - 1) // 2 == 231

match_path = match_features.main(
    matcher_conf, sfm_pairs, feature_path, matches=paths["matches"] / "matches-superpoint-lightglue.h5", overwrite=True
)
print("Features:", feature_path)
print("Exhaustive pairs:", pair_count)
print("Matches:", match_path)


## 5. Run HLoc/PyCOLMAP sparse Structure-from-Motion

Use `CameraMode.SINGLE` to share one camera. Initialize `SIMPLE_RADIAL` from COLMAP's default 1.2× maximum image dimension focal-length heuristic and centered principal point. Freeze all intrinsic parameters during bundle adjustment; this is a controlled prior because the phone's true focal length is unknown. The run performs HLoc's pair geometric verification before incremental mapping. No dense stage is invoked.


In [ ]:
INITIAL_CAMERA_PARAMS = [1920.0, 800.0, 450.0, 0.0]
image_options = pycolmap.ImageReaderOptions(
    camera_model="SIMPLE_RADIAL",
    camera_params=",".join(str(value) for value in INITIAL_CAMERA_PARAMS),
)
mapper_options = {
    "ba_refine_focal_length": False,
    "ba_refine_principal_point": False,
    "ba_refine_extra_params": False,
    "num_threads": min(os.cpu_count() or 1, 4),
}
print("Image reader options:", image_options.todict())
print("Mapper options:", mapper_options)

model = reconstruction.main(
    sfm_dir,
    IMAGE_DIR,
    sfm_pairs,
    feature_path,
    match_path,
    camera_mode=pycolmap.CameraMode.SINGLE,
    skip_geometric_verification=False,
    image_list=image_names,
    image_options=image_options,
    mapper_options=mapper_options,
)

BASELINE = {
    "registered_images": 4,
    "total_images": 22,
    "points_3d": 139,
    "camera_model": "SIMPLE_RADIAL",
    "camera_parameters": [3563.872745250371, 800.0, 450.0, -1.454033292928829],
    "mean_reprojection_error_px": 0.548383,
    "mean_track_length": 2.690647,
    "source": "data/terrain_dataset/reconstruction/logs/run-20261119T071852Z/reconstruction_report.json and diagnostics",
}
report = build_reconstruction_report(
    model, image_names, BASELINE, calibration_refined=False,
    initial_camera_params=INITIAL_CAMERA_PARAMS,
)
report["pipeline"] = {
    "feature_extractor": feature_conf,
    "preprocessing": preprocessing,
    "matcher": matcher_conf,
    "pairing": "exhaustive",
    "pair_count": pair_count,
    "camera_mode": "SINGLE",
    "mapper_options": mapper_options,
}
report["runtime"] = {
    "hloc_version": str(hloc.__version__),
    "pycolmap_version": str(pycolmap.__version__),
    "pytorch_version": str(torch.__version__),
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

if model is not None:
    model.write(str(paths["sparse"] / "0"))
    model.export_PLY(str(OUTPUT_ROOT / "points3D.ply"))
else:
    print("HLoc did not produce a reconstruction; no PLY was fabricated.")

report_path = save_reconstruction_report(report, OUTPUT_ROOT / "reconstruction_report.json")
print(json.dumps(report, indent=2))
print("Report:", report_path)


## 6. Visualize, compare, and download

The plot shows only reconstructed sparse points and registered camera centers returned by PyCOLMAP. An empty or tiny reconstruction remains empty or tiny; no terrain surface is synthesized.


In [ ]:
from IPython.display import display
import numpy as np
import plotly.graph_objects as go

print(f"Registered images: {report['registered_images']}/{report['input_images']}")
print("Unregistered images:", report["unregistered_images"])
print("3D points:", report["points_3d"])
print("Cameras:", report["camera_count"])
print("Mean / median reprojection error (px):", report["mean_reprojection_error_px"], "/", report["median_reprojection_error_px"])
print("Mean track length:", report["mean_track_length"])
print("Camera calibration:", report["cameras"])
print("Calibration refined:", report["calibration_policy"]["refined"])
print("Camera positions:", report["camera_positions"])

print("Baseline comparison:", {
    "registered_images": f"{report['registered_images']}/{report['input_images']} vs {BASELINE['registered_images']}/{BASELINE['total_images']}",
    "points_3d": f"{report['points_3d']} vs {BASELINE['points_3d']}",
    "mean_track_length": f"{report['mean_track_length']} vs {BASELINE['mean_track_length']}",
    "baseline_reprojection_error_px": BASELINE["mean_reprojection_error_px"],
    "note": "Calibration was fixed for HLoc; compare reprojection metrics cautiously. Review spatial distribution and intrinsics, not point count alone.",
})

if model is not None and report["points_3d"]:
    xyz = np.asarray([point.xyz for point in model.points3D.values()])
    rgb = np.asarray([point.color for point in model.points3D.values()])
    centers = np.asarray([[p["x"], p["y"], p["z"]] for p in report["camera_positions"]])
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
        marker={"size": 2, "color": [f"rgb({int(c[0])},{int(c[1])},{int(c[2])})" for c in rgb]},
        name="Sparse 3D points",
    ))
    if len(centers):
        fig.add_trace(go.Scatter3d(
            x=centers[:, 0], y=centers[:, 1], z=centers[:, 2], mode="markers+text",
            text=[p["image"] for p in report["camera_positions"]], textposition="top center",
            marker={"size": 6, "color": "red", "symbol": "diamond"}, name="Camera centers",
        ))
    fig.update_layout(title="HLoc sparse reconstruction (points and camera centers)", scene={"aspectmode": "data"})
    fig.show()
else:
    print("No sparse points available to visualize.")

ply_path = OUTPUT_ROOT / "points3D.ply"
if ply_path.exists():
    files.download(str(ply_path))
else:
    print("No PLY exists because no reconstruction was produced.")


## Output layout

```text
ares_hloc_outputs/
├── features/                  # SuperPoint HDF5
├── matches/                   # exhaustive pairs and LightGlue HDF5
├── reconstruction/            # HLoc database, COLMAP model, and logs
├── sparse/0/                  # exported COLMAP binary model
├── points3D.ply               # downloadable sparse point cloud
└── reconstruction_report.json # metrics, calibration, camera centers, baseline
```

The PLY contains sparse landmarks only. It is not a dense point cloud, mesh, or virtual terrain.
